# Customer Behavior Analysis – Alfido Tech
PREPARED BY:
Gannoju Sai Charani

**Objective:** Analyze customer transactions and behavior to identify customer segments, purchase patterns, retention trends and churn risks, then provide actionable recommendations to improve engagement.

**Dataset:** Kaggle – Customer Behavior Analysis
**Records:** 250,000 transactions
**Unique customers analyzed:** 49,673

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)

## 2. Load Dataset

In [ ]:
df = pd.read_csv('/content/ecommerce_customer_data_custom_ratios.csv.zip')
print('Dataset shape:', df.shape)
df.head()

## 3. Dataset Overview

In [ ]:
print('Rows and columns:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nData types and non-null counts:')
df.info()
print('\nSummary statistics:')
display(df.describe())

## 4. Data Cleaning

In [ ]:
print('Missing values before cleaning:')
display(df.isnull().sum())
print('Duplicate rows before cleaning:', df.duplicated().sum())

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Convert date to datetime
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'], errors='coerce')

# Returns is an indicator. Missing values are treated as no return for analysis.
df['Returns'] = df['Returns'].fillna(0)

# Remove rows missing essential analytical fields
df = df.dropna(subset=['Customer ID', 'Purchase Date', 'Total Purchase Amount']).copy()

print('Dataset shape after cleaning:', df.shape)
print('Missing values after cleaning:')
display(df.isnull().sum())

### Cleaning result

The source contains **47,596 missing Returns values (19.04%)** and no duplicate rows. The missing Returns indicators are filled with 0 (no return) so that return information can be used consistently.

## 5. Feature Engineering – RFM

In [ ]:
latest_date = df['Purchase Date'].max()

rfm = df.groupby('Customer ID').agg(
    Recency=('Purchase Date', lambda x: (latest_date - x.max()).days),
    Frequency=('Purchase Date', 'count'),
    Monetary=('Total Purchase Amount', 'sum')
).reset_index()

print('Latest transaction date:', latest_date)
print('Unique customers:', rfm['Customer ID'].nunique())
display(rfm.head())

### RFM meaning
- **Recency:** number of days since the customer last purchased. Lower is better.
- **Frequency:** number of transactions. Higher is better.
- **Monetary:** total purchase value. Higher is better.

## 6. RFM Scoring

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)

rfm['RFM_Total'] = rfm[['R_Score','F_Score','M_Score']].sum(axis=1)
display(rfm.head())

## 7. Customer Segmentation

In [ ]:
def segment(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 4:
        return 'Loyal Customers'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 3:
        return 'Potential Loyalists'
    elif row['R_Score'] <= 2 and row['M_Score'] >= 3:
        return 'At-Risk Customers'
    else:
        return 'Lost / Hibernating'

rfm['Segment'] = rfm.apply(segment, axis=1)
segment_counts = rfm['Segment'].value_counts()
segment_pct = (segment_counts / len(rfm) * 100).round(2)
segment_summary = rfm.groupby('Segment').agg(
    Customers=('Customer ID','count'),
    Average_Recency=('Recency','mean'),
    Average_Frequency=('Frequency','mean'),
    Average_Monetary=('Monetary','mean')
).sort_values('Customers', ascending=False)
segment_summary['Percentage'] = (segment_summary['Customers'] / len(rfm) * 100).round(2)
display(segment_summary)

In [ ]:
plt.figure(figsize=(10,5))
segment_counts.plot(kind='bar')
plt.title('Customer Segments')
plt.xlabel('Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 8. Segment Profiling

In [ ]:
segment_value = rfm.groupby('Segment')['Monetary'].mean().sort_values(ascending=False)
print('Average monetary value by segment:')
display(segment_value.to_frame('Average Monetary Value'))

plt.figure(figsize=(10,5))
segment_value.plot(kind='bar')
plt.title('Average Customer Value by Segment')
plt.xlabel('Segment')
plt.ylabel('Average Monetary Value')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 9. Purchase Pattern Analysis

In [ ]:
df['Month'] = df['Purchase Date'].dt.to_period('M')
monthly_sales = df.groupby('Month')['Total Purchase Amount'].sum()

plt.figure(figsize=(12,5))
monthly_sales.plot(kind='line', marker='o')
plt.title('Monthly Purchase Trends')
plt.xlabel('Month')
plt.ylabel('Total Purchase Amount')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Highest monthly purchase value:', monthly_sales.idxmax(), f'({monthly_sales.max():,.0f})')
print('Lowest monthly purchase value:', monthly_sales.idxmin(), f'({monthly_sales.min():,.0f})')
print('Total purchase value:', f'{monthly_sales.sum():,.0f}')

In [ ]:
category_sales = df.groupby('Product Category')['Total Purchase Amount'].sum().sort_values(ascending=False)
print('Purchase value by product category:')
display(category_sales.to_frame('Total Purchase Amount'))

plt.figure(figsize=(9,5))
category_sales.plot(kind='bar')
plt.title('Sales by Product Category')
plt.xlabel('Product Category')
plt.ylabel('Total Purchase Amount')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 10. Churn Analysis

In [ ]:
transaction_churn = df['Churn'].mean() * 100
customer_churn = df.groupby('Customer ID')['Churn'].max().mean() * 100
print(f'Transaction-level churn rate: {transaction_churn:.2f}%')
print(f'Customer-level churn rate: {customer_churn:.2f}%')

plt.figure(figsize=(7,5))
sns.countplot(data=df, x='Churn')
plt.title('Churn Distribution')
plt.xlabel('Churn (0 = No, 1 = Yes)')
plt.ylabel('Transactions')
plt.tight_layout()
plt.show()

## 11. High-Value Churn Risk

In [ ]:
recency_cutoff = rfm['Recency'].quantile(0.75)
monetary_cutoff = rfm['Monetary'].median()
at_risk = rfm[(rfm['Recency'] > recency_cutoff) & (rfm['Monetary'] > monetary_cutoff)].copy()
print('High-value behaviorally at-risk customers:', len(at_risk))
display(at_risk.head())

## 12. Retention Analysis

In [ ]:
purchase_counts = df.groupby('Customer ID').size()
repeat_customers = purchase_counts[purchase_counts > 1]
repeat_rate = len(repeat_customers) / len(purchase_counts) * 100
print('Total customers:', len(purchase_counts))
print('Repeat customers:', len(repeat_customers))
print(f'Repeat customer rate: {repeat_rate:.2f}%')

# Monthly repeat-customer rate
tx = df[['Customer ID','Purchase Date']].copy()
tx['Month'] = tx['Purchase Date'].dt.to_period('M')
first_month = tx.groupby('Customer ID')['Month'].min().rename('First_Month')
tx = tx.join(first_month, on='Customer ID')
tx['Repeat'] = tx['Month'] > tx['First_Month']
retention = tx.groupby('Month').agg(Active_Customers=('Customer ID','nunique'), Repeat_Customers=('Repeat','sum'))
retention['Repeat_Customer_Rate'] = retention['Repeat_Customers'] / retention['Active_Customers']

plt.figure(figsize=(12,5))
retention['Repeat_Customer_Rate'].plot(marker='o')
plt.title('Monthly Repeat-Customer Rate')
plt.xlabel('Month')
plt.ylabel('Repeat Customer Rate')
plt.ylim(0,1)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 13. Key Findings from the Actual Dataset

- **49,673 unique customers** were identified from **250,000 transactions**.
- **96.61%** of customers are repeat purchasers (47,988 customers).
- Customer-level churn is approximately **20.01%**.
- The largest segment is **Lost / Hibernating: 16,316 customers (32.85%)**.
- **Champions:** 8,692 customers (17.50%) with the highest average monetary value, about **21,789.39**.
- **Loyal Customers:** 6,268 (12.62%), average monetary value about **17,458.95**.
- **At-Risk Customers:** 9,049 (18.22%), average monetary value about **16,714.46**, making them an important win-back opportunity.
- **Potential Loyalists:** 9,348 (18.82%).
- **Books** generated the highest purchase value among categories, followed closely by Clothing.
- Total purchase value across the dataset is approximately **681.34 million**.
- The highest monthly purchase value occurred in **December 2020 (16.29 million)**; September 2023 is lower because the dataset ends part-way through that month.

## 14. Five Actionable Recommendations for Alfido Tech

1. **Protect Champions:** create loyalty tiers, exclusive benefits, early access and personalized cross-sell recommendations.
2. **Win back At-Risk Customers:** launch automated reactivation campaigns with personalized discounts, reminders and time-limited offers.
3. **Convert Potential Loyalists:** encourage the next purchase using bundles, product recommendations and second-purchase incentives.
4. **Reactivate Lost / Hibernating Customers:** use targeted low-cost win-back campaigns and reduce spend on customers who remain unresponsive.
5. **Use category and timing insights:** prioritize Books and Clothing where purchase value is strongest and schedule campaigns using observed monthly purchase patterns.

## 15. Conclusion

RFM segmentation shows that Alfido Tech has a strong repeat-purchase base but also a large Lost / Hibernating group and a meaningful At-Risk high-value group. The best opportunity is to protect Champions and Loyal Customers while using targeted, personalized reactivation strategies for At-Risk and inactive customers. Combining RFM, churn and purchase-trend insights can make engagement campaigns more focused and measurable.

## 16. Export Results

In [ ]:
rfm.to_csv('alfido_customer_segments.csv', index=False)
segment_summary.to_csv('alfido_segment_summary.csv')
retention.to_csv('alfido_retention.csv')
print('Exported: alfido_customer_segments.csv, alfido_segment_summary.csv, alfido_retention.csv')